# PI-10 UAT
This notebook addresses requirements in []

In [1]:
import geopandas as gpd
import pandas as pd
import xarray as xr

In [2]:
nhf ="../data/nhf_1.2.2.gpkg"

## Reservoirs

- Show that NWM v3 RFC and USACE gages/lakes are present.
- Show that Ohio RFC gages addedand crosswalked to reservoirs where possible
- Expanded NHF reservoir set including USBR reservoirs, with reservoir DA applied where available and supplied to t-route. Additional reservoirs cross-walked to available gages.

In [18]:
res_index = "../data/sconus/lakes/input/reservoir_index_AnA.nc"
active_rfc = "../data/sconus/lakes/input/nwps_all_gauges_report.csv"
ohio_rfc = "../data/sconus/lakes/input/adhoc_lakes.gpkg"

res_da = gpd.read_file(nhf, layer="reservoir_da")
lakes = gpd.read_file(nhf, layer="lakes")
gages = gpd.read_file(nhf, layer="gages")
ds_res_index = xr.open_dataset(res_index)
df_active_rfc = pd.read_csv(active_rfc)
df_ohio_rfc = gpd.read_file(ohio_rfc)

usace_gage_id_field: str = "usace_gage_id"
usace_lake_id_field: str = "usace_lake_id"
rfc_gage_id_field: str = "rfc_gage_id"
rfc_lake_id_field: str = "rfc_lake_id"
active_gage_id: str = "nws shef id"

In [ ]:
# Helper functions for comparison of NWM v3 lakes
def get_crosswalk(ds: xr.Dataset, gage_field, lake_field, output_gage_field: str = 'site_no', lake_id_field: str = 'lake_id', active_rfc: pd.DataFrame | None = None):
    """
    Retrieve a crosswalk for a given gage type. Returns gage : lake_id
    For RFC gages, filters by active gages.
    """
    crosswalk = pd.DataFrame(
                data={
                    gage_field: ds[gage_field].to_numpy(),
                    lake_field: ds[lake_field].to_numpy(),
                }
            )
    crosswalk[gage_field] = crosswalk[gage_field].apply(lambda x: x.decode("utf-8")).str.strip()


    # for RFC gages, filter by active gages if the data is available
    if gage_field == rfc_gage_id_field and active_rfc is not None:
        crosswalk = crosswalk.loc[crosswalk[gage_field].isin(active_rfc[active_gage_id])].copy()

    crosswalk.rename(columns={gage_field: output_gage_field, lake_field: lake_id_field}, inplace=True)

    return crosswalk

def assert_gage_present(res_type: str, crosswalk: pd.DataFrame, gages: gpd.GeoDataFrame, gage_id_field: str ='site_no'):
    """Compares crosswalk list to gages layer. Returns missing gages."""
    missing_gages = crosswalk.loc[~crosswalk[gage_id_field].astype(str).isin(gages[gage_id_field])].copy()
    len_missing = len(missing_gages)
    print(f"{len_missing} {res_type} missing gages in gage layer")
    return missing_gages

def assert_lake_present(res_type: str, crosswalk: pd.DataFrame, lakes: gpd.GeoDataFrame, lake_id_field: str ='lake_id'):
    """Compares crosswalk list to lakes layer. Returns missing lakes."""
    missing_lakes = crosswalk.loc[~crosswalk[lake_id_field].astype(str).isin(lakes[lake_id_field])].copy()
    len_missing = len(missing_lakes)
    print(f"{len_missing} {res_type} missing lakes in lakes layer")
    return missing_lakes            


### NWM v3 USACE
Obtain the USACE crosswalk, return any missing gages, return any missing lakes.

In [5]:
crosswalk_usace = get_crosswalk(ds_res_index, usace_gage_id_field, usace_lake_id_field)

In [6]:
# NOTE: Expected output for `assert_gage_present` will have missing `PortHuron` gage. 
# This gage is represented in NHF as USGS `04159130`. The lake is not missing.
assert_gage_present('USACE', crosswalk_usace,  gages)

1 USACE missing gages in gage layer


,site_no,lake_id
44,PortHuron,4800004


In [7]:
assert_lake_present('USACE', crosswalk_usace, lakes)

0 USACE missing lakes in lakes layer


,site_no,lake_id


### NWM v3 RFC
Obtain the RFC crosswalk, return any missing gages, return any missing lakes.

NOTE: The NWM v3 RFC gages were filtered by active NWS gage because non-active RFC gages were missing from OWP API and shown as discontinued on USGS. Flows would not be available so they were not included. 97 RFC gages were dropped. These gages were also not present in HF 2.2.

In [16]:
crosswalk_rfc = get_crosswalk(ds_res_index, rfc_gage_id_field, rfc_lake_id_field, active_rfc=df_active_rfc)

In [17]:
assert_gage_present('RFC', crosswalk_rfc, gages)

0 RFC missing gages in gage layer


,site_no,lake_id


In [10]:
assert_lake_present('RFC', crosswalk_rfc, lakes)

0 RFC missing lakes in lakes layer


,site_no,lake_id


### Ohio RFC
Ohio RFC stations were added and crosswalked to lakes for 76/80 stations. The 4 missing stations were manually inspected and were not near a lake.

In [23]:
missing_gages = df_ohio_rfc.loc[~df_ohio_rfc['locationId'].isin(gages['site_no'])].copy()
print(f"{len(missing_gages)} Ohio RFC missing gages in gage layer")
missing_gages

0 Ohio RFC missing gages in gage layer


,wkt_geom,filename,Source File Identifier,locationId,stationName,Lat,Lon,lake_id,site_no,dam_id,confidence,comments,ref_waterbodies_only,geometry


In [ ]:
# Expected: 4 missing null lakes
df_ohio_rfc['lake_id']=df_ohio_rfc['lake_id'].astype(str)
df_ohio_rfc.replace('-99999', None, inplace=True)
missing_lakes = df_ohio_rfc.loc[~df_ohio_rfc['lake_id'].isin(lakes['lake_id'])].copy()
print(f"{len(missing_lakes)} Ohio RFC missing lakes in lakes layer")
missing_lakes

4 Ohio RFC missing lakes in lakes layer


,wkt_geom,filename,Source File Identifier,locationId,stationName,Lat,Lon,lake_id,site_no,dam_id,confidence,comments,ref_waterbodies_only,geometry
4,Point (-84.17000000000000171 39.07000000000000...,202412172017_OHRFC_Reservoir_Export_SMIM.xml,SMIM,BHDO1OT,Harsha Dam,39.07000,-84.1700,None,3247050,ls-50016,-99999,"lake missing, gage far, 2 dams in close proxim...",NaN,POINT (-84.17 39.07)
6,Point (-81.4328000000000003 40.64860999999999791),202412172011_OHRFC_Reservoir_Export_SMKU.xml,SMKU,BIVO1OT,Bolivar Lake,40.64861,-81.4328,None,-99999,ls-54794,-99999,gage and lake missing,NaN,POINT (-81.4328 40.64861)
47,Point (-83.25749999999999318 36.75222000000000...,202412172022_OHRFC_Reservoir_Export_SCMU.xml,SCMU,MNLK2OT,Smith,36.75222,-83.2575,None,3400800,ls-52548,-99999,lake missing,NaN,POINT (-83.2575 36.75222)
74,Point (-79.81999999999999318 41.89999999999999...,202412172008_OHRFC_Reservoir_Export_SAGU.xml,SAGU,UCDP1,Union City,41.90000,-79.8200,None,-99999,ls-57097,-99999,"gage and lake missing, dam far",NaN,POINT (-79.82 41.9)


### USBR Reservoirs

In [20]:
df_active_rfc.columns

Index(['location name', 'proximity', 'river/water-body name', 'nws shef id',
       'location type', 'usgs id', 'latitude', 'longitude', 'wfo', 'rfc',
       'state', 'county', 'wrr', 'timezone', 'inundation', 'elevation',
       'action stage', 'flood stage', 'moderate flood stage',
       'major flood stage', 'flood stage unit', 'coeid', 'hydrograph page',
       'pedts', 'in service', 'hemisphere',
       'low water threshold value / units', 'forecast status',
       'display low water impacts', 'low flow display',
       'give data attribution', 'attribution wording', 'fema wms',
       'probabilistic site', 'weekly chance probabilistic enabled',
       'short-term probabilistic enabled',
       'chance of exceeding probabilistic enabled',
       'nrldb vertical datum name', 'nrldb vertical datum',
       'navd88 vertical datum', 'ngvd29 vertical datum', 'msl vertical datum',
       'other vertical datum', 'reach id'],
      dtype='object')